# Initial results of creating knowledge graphs for research papers

## Steps 1 & 2: Pip Installs And Imports

In [58]:
# Step 1: Do pip installs
packages = ["networkx", "pyvis", "PyMuPDF",
            "requests", "spacy", "regex",
            "keybert", "transformers", "sentence-transformers",
            "nltk"]

def install(package: str):
  !uv pip install -q {package}

for package in packages:
  install(package)

print("Installs finished!")

Installs finished!


In [59]:
# Step 2: All necessary imports
import networkx as nx
from pyvis.network import Network
import fitz # PyMuPDF
import requests
import spacy
import itertools
from collections import Counter
import re
from keybert import KeyBERT

from transformers import pipeline
from itertools import combinations
from collections import defaultdict

from sentence_transformers import SentenceTransformer, util
import numpy as np

from IPython.display import HTML

In [60]:
# Download spaCy model only if not already installed
import spacy.util
model_name = 'en_core_web_sm'
if not spacy.util.is_package(model_name):
    # This will always get the latest version
    !uv pip install -q spacy
    !python -m spacy download en_core_web_sm

## Step 3: Text Extraction

In [61]:
def pdf_to_text(path: str) -> str:
    with fitz.open(path) as doc:
        return "\n".join(page.get_text() for page in doc)

def load_paper_text_from_url(paper_url: str, paper_filename: str) -> str:
  response = requests.get(paper_url)
  with open(paper_filename, "wb") as f:
    f.write(response.content)

  return pdf_to_text(paper_filename)

In [62]:
try:
    import google.colab
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Not running in Google Colab. Skipping file upload.")

def load_paper_text_from_file(paper_path: str):
  doc = fitz.open(paper_path)
  text = ""
  for page in doc:
      text += page.get_text()
  return text

Not running in Google Colab. Skipping file upload.


In [63]:
# paper_url = "https://zhenlab.com/wp-content/uploads/2025/02/Transfer_learning_paper___Bioinformatics_Advances.pdf"
# paper_filename = "zhenlab_paper.pdf"
# paper_text = load_paper_text_from_url(paper_url, paper_filename)

# paper_path = "Can MOOC Instructor Be Portrayed by Semantic Features.pdf"
# paper_text = load_paper_text_from_file(paper_path)
# print(f"Sample text from paper:\n{paper_text[:500]}...")

## Step 4: Creation of Nodes

In [64]:
# load multiple papers and store them in a dictionary
def load_multiple_papers(paper_paths: list):
    """
    Load multiple papers and return a dictionary with paper path as key and text as value
    """
    papers = {}
    for paper_path in paper_paths:
        try:
            papers[paper_path] = load_paper_text_from_file(paper_path)
            print(f"Successfully loaded: {paper_path}")
        except Exception as e:
            print(f"Failed to load {paper_path}: {e}")
    
    return papers

In [65]:
try:
    import google.colab
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Not running in Google Colab. Skipping file upload.")

def load_paper_text_from_file(paper_path: str):
  doc = fitz.open(paper_path)
  text = ""
  for page in doc:
      text += page.get_text()
  return text

Not running in Google Colab. Skipping file upload.


In [66]:
# Step 4: Creates nodes
# Note this is a section of the code where there are multiple strategies
# to do this step. Some examples include:
#     - Named-Entity Recognition
#     - Extract noun phrases
#     - Extract ranked phrases with KeyBERT

# Modified to track paper source with each node
def baseline_create_nodes(paper_text: str, node_limit: int, paper_source: str) -> list:
  # The following does noun phrase extraction
  nlp = spacy.load("en_core_web_sm")
  doc = nlp(paper_text)
  noun_phrases = [chunk.text.lower().strip() for chunk in doc.noun_chunks]

  # Reduce number of nodes, required for successful rendering
  noun_phrases = [np for np in noun_phrases if len(np.split()) > 1 and len(np) > 3]
  phrase_counts = Counter(noun_phrases)

  filtered_phrases = [phrase for phrase, _ in phrase_counts.most_common(node_limit)]
  nodes = list(set(filtered_phrases))
  # Return nodes with source information
  nodes_with_source = [(node, paper_source) for node in nodes]
  
  return nodes_with_source

In [67]:
# More summary focused method of creating nodes - modified to track paper source

def extract_sections(text):
    section_keywords = ['abstract', 'introduction', 'background', 'related work', 'method', 'methods',
                        'materials', 'experiments', 'results', 'discussion', 'conclusion', 'conclusions'] # won't put references, extra gibberish
    section_pattern = r'(?i)^(' + '|'.join(re.escape(k) for k in section_keywords) + r')$'

    lines = text.split('\n')
    sections = {}
    current_section = None
    for line in lines:
        stripped = line.strip().lower()
        if re.match(section_pattern, stripped):
            current_section = stripped
            print("Found section: " + current_section)
            sections[current_section] = []
        elif current_section:
            sections[current_section].append(stripped)

    # Join lines per section
    return {k: "\n".join(v).strip() for k, v in sections.items() if v}

def summarize_sections(sections, max_input_tokens=1500, max_output_tokens=300):
    summarized = {}
    for name, content in sections.items():
        # Skip very short sections
        if len(content.strip().split()) < 10:
            continue

        # Truncate content to avoid token limit issues
        truncated = content.strip().replace("\n", " ")[:4000]

        try:
            summary = summarizer(
                truncated,
                max_length=max_output_tokens,
                min_length=100,
                do_sample=False
            )
            summarized[name] = summary[0]['summary_text']
        except Exception as e:
            print(f"Failed to summarize section '{name}': {e}")

    return summarized

def extract_phrases_from_summaries(summaries, phrases_per_section=20):
    all_phrases = []
    for section, text in summaries.items():
        # Extract keywords/phrases using KeyBERT
        keywords = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 3),
            stop_words='english',
            use_maxsum=True,
            nr_candidates=20, # number of noun phrases returned for each summary
            top_n=phrases_per_section
        )
        section_phrases = [kw for kw, score in keywords]
        all_phrases.extend(section_phrases)

    return all_phrases

def get_top_nodes(phrases, limit=100):
    phrase_counts = Counter(phrases)
    return [phrase for phrase, _ in phrase_counts.most_common(limit)]

def summary_based_create_nodes(paper_text: str, node_limit: int, paper_source: str) -> list:
  phrases = extract_phrases_from_summaries(summaries)

  if len(phrases) < node_limit:
    node_limit = len(phrases)

  nodes = get_top_nodes(phrases, node_limit)
  # Return nodes with source information
  nodes_with_source = [(node, paper_source) for node in nodes]
  return nodes_with_source

## Step 5: Creation of Edges

In [68]:
# Modified to work with nodes that include source information
def create_edges_by_sentence_coocurrence(nodes_with_source, papers_dict):
  nlp = spacy.load("en_core_web_sm")
  edge_weights = defaultdict(int)

  # Extract just the node texts for matching
  node_texts = [node[0] for node in nodes_with_source]
  
  # Process each paper
  for paper_path, paper_text in papers_dict.items():
    doc = nlp(paper_text)
    sentences = [sent.text.lower() for sent in doc.sents]
    
    for sentence in sentences:
      # Find which nodes are present in this sentence
      present_nodes = [node_idx for node_idx, (node, _) in enumerate(nodes_with_source) if node in sentence]
      for idx1, idx2 in itertools.combinations(set(present_nodes), 2):
        # Use indices to get the original nodes with source
        node1, node2 = nodes_with_source[idx1], nodes_with_source[idx2]
        # Sort by node text to ensure consistent edge representation
        if node1[0] > node2[0]:
            node1, node2 = node2, node1
        edge = (node1, node2)
        edge_weights[edge] += 1

  edges = [(a, b, {'weight': w}) for (a, b), w in edge_weights.items()]
  return edges

In [69]:
def create_edges_by_summary_cooccurrence(nodes_with_source, papers_dict):
    edges = defaultdict(int)
    
    # For each paper, extract sections and summaries
    for paper_path, paper_text in papers_dict.items():
        sections = extract_sections(paper_text)
        summaries = summarize_sections(sections)
        
        for summary in summaries.values():
            summary_lower = summary.lower()
            # Find which nodes are present in this summary
            present_nodes = [node_with_source for node_with_source in nodes_with_source 
                            if node_with_source[0] in summary_lower]
            
            for node1, node2 in combinations(set(present_nodes), 2):
                # Sort by node text to ensure consistent edge representation
                if node1[0] > node2[0]:
                    node1, node2 = node2, node1
                edges[(node1, node2)] += 1  # Increment weight if they co-occur

    return [(a, b, {'weight': w}) for (a, b), w in edges.items()]

In [70]:
def create_edges_by_embedding_similarity(nodes_with_source, threshold=0.5):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    # Extract just the node texts for embedding
    node_texts = [node[0] for node in nodes_with_source]
    embeddings = model.encode(node_texts, convert_to_tensor=True)

    edges = []
    for i in range(len(nodes_with_source)):
        for j in range(i + 1, len(nodes_with_source)):
            sim = util.cos_sim(embeddings[i], embeddings[j]).item()
            if sim >= threshold:
                edges.append((nodes_with_source[i], nodes_with_source[j], {'weight': sim}))
    return edges

## Step 6: Populate Graph

In [71]:
# Step 6: Populate graph - modified to work with nodes that include source information

def create_graph(nodes_with_source, edges, paper_colors=None):
  graph = nx.Graph()
  
  # If no colors provided, generate random distinct colors
  if paper_colors is None:
    import random
    all_sources = set(source for _, source in nodes_with_source)
    paper_colors = {source: f"#{random.randint(0, 0xFFFFFF):06x}" for source in all_sources}

  # Add nodes with source information as attributes
  for node_text, source in nodes_with_source:
    graph.add_node(node_text, source=source, color=paper_colors.get(source, "#808080"), 
                   title=f"Source: {source}")

  # Add edges
  for (node1_with_source, node2_with_source, weight) in edges:
    node1_text, _ = node1_with_source
    node2_text, _ = node2_with_source
    graph.add_edge(node1_text, node2_text, weight=weight['weight'])

  print(f"Number of nodes: {len(graph.nodes)}")
  print(f"Number of edges: {len(graph.edges)}\n")
  print(f"Sources: {set(nx.get_node_attributes(graph, 'source').values())}")

  return graph

## Steps 7 & 8: Write Out And Display Graph

In [74]:
def graph_to_html(graph, path: str, display: False):
    net = Network(height="750px", width="100%", notebook=True, cdn_resources="in_line")
    
    # Configure network to show node information on hover
    net.set_options("""
    {
      "nodes": {
        "font": {
          "size": 15,
          "face": "Tahoma"
        }
      },
      "edges": {
        "color": {
          "inherit": true
        },
        "smooth": false
      },
      "physics": {
        "barnesHut": {
          "gravitationalConstant": -80000,
          "springLength": 250,
          "springConstant": 0.001
        },
        "minVelocity": 0.75
      }
    }
    """)
    
    # Add nodes with source information
    for node, data in graph.nodes(data=True):
        title = f"Source: {data.get('source', 'Unknown')}"
        net.add_node(node, color=data.get('color', '#808080'), title=title, label=node)
    
    # Add edges
    for source, target, data in graph.edges(data=True):
        net.add_edge(source, target, value=data.get('weight', 1))
    
    # Write HTML with UTF-8 encoding to avoid UnicodeEncodeError on Windows
    html_str = net.generate_html()
    with open(path, "w", encoding="utf-8") as f:
        f.write(html_str)

    if display:
        from IPython.display import HTML
        with open(path, "r", encoding="utf-8") as f:
            html_content = f.read()
        display(HTML(html_content))

# Complete Multi-Paper Knowledge Graph Generator

In [ ]:
# Define paper paths
paper_paths = [
    "pdfs/Transfer learning improves performance in volumetric.pdf",
    "pdfs/Automated Cell Structure Extraction for 3D Electron.pdf"
]

# Define colors for each paper
paper_colors = {
    paper_paths[0]: "#3366CC",  # Blue for first paper
    paper_paths[1]: "#DC3912"   # Red for second paper
}

# Load papers
papers_dict = load_multiple_papers(paper_paths)

# Create nodes for each paper (limit nodes per paper to avoid overcrowding)
nodes_per_paper = 25
combined_nodes = []

# Option 1: Baseline node creation
# for paper_path, paper_text in papers_dict.items():
#     paper_nodes = baseline_create_nodes(paper_text, nodes_per_paper, paper_path)
#     combined_nodes.extend(paper_nodes)
#     print(f"Created {len(paper_nodes)} nodes from {paper_path}")

# Option 2: Summary-based node creation (comment out Option 1 if using this)
summarizer = pipeline("summarization")
kw_model = KeyBERT()
for paper_path, paper_text in papers_dict.items():
    sections = extract_sections(paper_text)
    summaries = summarize_sections(sections)
    paper_nodes = summary_based_create_nodes(paper_text, nodes_per_paper, paper_path)
    combined_nodes.extend(paper_nodes)
    print(f"Created {len(paper_nodes)} nodes from {paper_path}")

# Create edges
# You can choose one of these edge creation methods
# edges = create_edges_by_sentence_coocurrence(combined_nodes, papers_dict)
# edges = create_edges_by_summary_cooccurrence(combined_nodes, papers_dict)
edges = create_edges_by_embedding_similarity(combined_nodes, threshold=0.6)

# Create graph
multi_paper_graph = create_graph(combined_nodes, edges, paper_colors)

# Export graph to HTML
graph_to_html(multi_paper_graph, "multi_paper_knowledge_graph_25_nodes.html", False)

print(f"Total nodes: {len(combined_nodes)}")
print(f"Total edges: {len(edges)}")

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.


Successfully loaded: pdfs/Transfer learning improves performance in volumetric.pdf
Successfully loaded: pdfs/Automated Cell Structure Extraction for 3D Electron.pdf


Device set to use cpu


Found section: abstract
Found section: introduction
Found section: results
Found section: discussion
Found section: method
Found section: experiments
Found section: methods


Your max_length is set to 300, but your input_length is only 266. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=133)
Your max_length is set to 300, but your input_length is only 272. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=136)
Your max_length is set to 300, but your input_length is only 272. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=136)


Created 25 nodes from pdfs/Transfer learning improves performance in volumetric.pdf
Found section: abstract
Found section: introduction
Found section: results
Found section: discussion
Found section: background
Created 25 nodes from pdfs/Automated Cell Structure Extraction for 3D Electron.pdf
Created 25 nodes from pdfs/Automated Cell Structure Extraction for 3D Electron.pdf
Number of nodes: 50
Number of edges: 169

Sources: {'pdfs/Transfer learning improves performance in volumetric.pdf', 'pdfs/Automated Cell Structure Extraction for 3D Electron.pdf'}
Total nodes: 50
Total edges: 169
Number of nodes: 50
Number of edges: 169

Sources: {'pdfs/Transfer learning improves performance in volumetric.pdf', 'pdfs/Automated Cell Structure Extraction for 3D Electron.pdf'}
Total nodes: 50
Total edges: 169


## Analysis Functions for Multi-Paper Knowledge Graphs

In [86]:
def analyze_multi_paper_graph(graph):
    """Analyze the multi-paper knowledge graph to identify connections between papers"""
    # Get node sources
    node_sources = nx.get_node_attributes(graph, 'source')
    
    # Count nodes by source
    source_counts = Counter(node_sources.values())
    print("Nodes per paper:")
    for source, count in source_counts.items():
        print(f"  {source}: {count} nodes")
    
    # Find cross-paper connections
    cross_paper_edges = []
    for u, v in graph.edges():
        if node_sources[u] != node_sources[v]:
            cross_paper_edges.append((u, v))
    
    print(f"\nCross-paper connections: {len(cross_paper_edges)} edges")
    if cross_paper_edges:
        print("Sample cross-paper connections:")
        for i, (u, v) in enumerate(cross_paper_edges[:5]):  # Show up to 5 examples
            print(f"  {u} ({node_sources[u]}) <--> {v} ({node_sources[v]})")
    
    # Find highest-degree nodes
    degrees = dict(graph.degree())
    top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop connected concepts:")
    for node, degree in top_nodes:
        print(f"  {node} ({node_sources[node]}): {degree} connections")
            
    return {
        'source_counts': source_counts,
        'cross_paper_edges': cross_paper_edges,
        'top_nodes': top_nodes
    }

# Example usage
analysis = analyze_multi_paper_graph(multi_paper_graph)

Nodes per paper:
  pdfs/Transfer learning improves performance in volumetric.pdf: 25 nodes
  pdfs/Automated Cell Structure Extraction for 3D Electron.pdf: 25 nodes

Cross-paper connections: 65 edges
Sample cross-paper connections:
  microscopy (pdfs/Transfer learning improves performance in volumetric.pdf) <--> cell images (pdfs/Automated Cell Structure Extraction for 3D Electron.pdf)
  microscopy (pdfs/Transfer learning improves performance in volumetric.pdf) <--> electron microscope (pdfs/Automated Cell Structure Extraction for 3D Electron.pdf)
  microscopy (pdfs/Transfer learning improves performance in volumetric.pdf) <--> analysis cell images (pdfs/Automated Cell Structure Extraction for 3D Electron.pdf)
  microscopy (pdfs/Transfer learning improves performance in volumetric.pdf) <--> scanning electron microscope (pdfs/Automated Cell Structure Extraction for 3D Electron.pdf)
  electron microscopy (pdfs/Transfer learning improves performance in volumetric.pdf) <--> electron microsc